In [ ]:
#LangGraph构建具有长期记忆的智能体
# Lesson 2: Baseline Email Assistant

In [ ]:
# 加载 .env 文件里的环境变量（OPENAI_API_KEY 等），后面 init_chat_model / ChatOpenAI 会自动读取
import os
from dotenv import load_dotenv
_=load_dotenv()

## Setup a Profile, Prompt and Example Email

In [ ]:
# 用户画像（会被塞进各种 prompt 模板里），模拟“John”这个被服务的用户
profile = {
    "name": "John",                 # 邮件里对他的称呼
    "full_name": "John Doe",        # 全名，用于更正式的场合
    "user_profile_background": "Senior software engineer leading a team of 5 developers",  # 背景信息，帮助 LLM 判断邮件重要性
}

In [ ]:
# 分诊（triage）规则 + 主 agent 的行为指令，都是自然语言写的“提示词片段”
# 会被填入 prompts.py 里的模板，用于指导 LLM 如何对邮件分类、如何使用工具
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",   # 直接忽略的邮件类型
        "notify": "Team member out sick, build system notifications, project status updates",  # 需要提醒但不用回复
        "respond": "Direct questions from team members, meeting requests, critical bug reports",  # 需要直接回复
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently."
}

In [ ]:
# 一封示例邮件，用来跑通"分诊 -> 回复"整个流程
# Example incoming email
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

## Define the first part of the agent - triage.


In [ ]:
# BaseModel/Field 用来定义结构化输出的 schema（Router）
# TypedDict 用来定义 LangGraph 的 State（图节点间传递的共享状态）
# init_chat_model 是 LangChain 统一的模型加载入口，支持 "provider:model_name" 字符串写法
from pydantic import BaseModel ,Field
from typing_extensions import TypedDict,Literal,Annotated
from langchain.chat_models import init_chat_model

In [ ]:
# 用于"分诊"分类任务的小模型（便宜快速即可，不需要很强的推理能力）
llm=init_chat_model("openai:gpt-4o-mini")

In [ ]:
# Router：定义 LLM 结构化输出的 schema。
# 配合 with_structured_output() 使用，LLM 会被强制按这个 pydantic 模型的字段返回结果，
# 而不是返回自由文本，这样后面可以直接用 result.classification 做条件路由（if/elif）。
#
# TODO: 请在此处补全代码
# 定义一个继承自 BaseModel 的 Router 类，需要两个字段：
#   - reasoning: str，描述分类的推理过程
#   - classification: Literal["ignore", "respond", "notify"]，三选一的分类结果
# 别忘了给每个字段写 Field(description=...)，这些描述会影响 LLM 的输出质量。

In [ ]:
# 把 llm 包装成"结构化输出"版本：调用 .invoke() 时返回值直接是 Router 的实例
llm_router=llm.with_structured_output(Router)

In [ ]:
# 从同目录的 prompts.py 导入提示词模板
# 注意：原始课程素材里应附带 prompts.py，但这份目录里缺失了该文件（import 会直接
# ModuleNotFoundError，后面所有代码都跑不动）。这里已按 lesson_4/lesson_5 中内联的
# triage_system_prompt 原文、以及各处 .format(author=..., to=..., subject=...,
# email_thread=...) 的调用方式，补上了一份 prompts.py（含 triage_system_prompt /
# triage_user_prompt / agent_system_prompt）。
from prompts import triage_system_prompt, triage_user_prompt

In [ ]:
# 用 profile / prompt_instructions 把 triage_system_prompt 模板里的占位符填满
# examples=None：这一课还没有引入"少样本示例记忆"（那是 lesson_4 才讲的内容），先留空
system_prompt=triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

In [ ]:
# 把具体这封邮件的信息填进 triage_user_prompt，作为 user message 发给 LLM
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"],
)

In [ ]:
# 调用结构化输出模型做一次分类推理
# 注意：这里会真正发起网络请求（调用 OpenAI API）。当前 .env 里是占位的假 key，
# 所以这一步预期会因为鉴权失败而报错——这是正常现象，不是代码逻辑问题。
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

In [ ]:
# 打印分类结果（一个 Router 实例，包含 reasoning 和 classification 两个字段）
print(result)

## Main agent, define tools

In [ ]:
# @tool 装饰器：把普通 Python 函数注册成 LangChain 工具，供 ReAct agent 在推理过程中调用
from langchain_core.tools import tool

In [ ]:
# 工具 1：发送邮件（这里只是占位实现，真实项目里会接真正的邮件发送 API）
# docstring 会被 LLM 读到，用来判断"什么时候该调用这个工具"，所以要写清楚
@tool
def write_email(to:str,subject:str,content:str)->str:
    """Write and send an email."""
    # TODO: 请在此处补全代码
    # 返回一个字符串，说明邮件已发送给谁、主题是什么（占位实现即可，不需要真的发邮件）
    pass

In [ ]:
# 工具 2：安排会议（同样是占位实现）
@tool
def schedule_meeting(
        attendees:list[str],
        subject:str,
        duration_minutes:int,
        preferred_day:str
)->str:
        """Schedule a calendar meeting."""
        # TODO: 请在此处补全代码
        # 返回一个字符串，说明会议主题、安排在哪天、有几位参会者
        pass

In [ ]:
# 工具 3：查看某天的日历可用时段
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # TODO: 请在此处补全代码
    # 返回一个字符串，列出该天的几个可用时间段（占位数据即可）
    pass

## Main agent, define prompt

In [ ]:
# 从 prompts.py 导入主 agent 的系统提示词模板
from prompts import agent_system_prompt

# create_prompt 是一个"动态生成 prompt 的函数"，会被传给 create_react_agent 的 prompt 参数。
# 每次 agent 推理前都会调用它，用当前 state 拼出完整的消息列表：
# [系统消息] + 历史对话消息（state['messages']）
#
# TODO: 请在此处补全代码
# 实现 create_prompt(state)：
#   1. 用 agent_system_prompt.format(instructions=..., **profile) 生成系统提示词内容
#   2. 返回 [{"role": "system", "content": ...}] + state['messages']
def create_prompt(state):
    pass

In [ ]:
# 打印模板原文，看看里面有哪些占位符
print(agent_system_prompt)

In [ ]:
# create_react_agent：LangGraph 预置的 ReAct 模式 agent 构造函数。
# 它内部会构建一个"LLM 决策 -> 调用工具 -> 把结果喂回 LLM -> 再决策"的循环子图，
# 直到 LLM 判断不需要再调用工具为止。
from langgraph.prebuilt import create_react_agent

In [ ]:
# agent 可以使用的全部工具列表
tools=[write_email,schedule_meeting,check_calendar_availability]

In [ ]:
# 构建主 agent："openai:gpt-4o" 做实际的回复/工具调用推理（比分诊模型更强）
# prompt=create_prompt：每一步都用上面定义的函数动态生成系统提示词
# TODO: 请在此处补全代码
# 用 create_react_agent(model, tools=tools, prompt=create_prompt) 构建 agent
agent = None

In [ ]:
# 单独测试一下主 agent：让它回答"周二有空吗"，预期它会调用 check_calendar_availability 工具
# 同样会真正调用 OpenAI API，用假 key 预期在此报鉴权错误
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "what is my availability for tuesday?"
    }]}
)

In [ ]:
# pretty_print()：把最后一条消息（agent 的最终回复）用可读格式打印出来
response["messages"][-1].pretty_print()

## Create the Overall Agent

**架构说明**：接下来要把"分诊"和"主 agent"组装成一张 LangGraph 图（StateGraph）：

1. `State`：图里各节点共享、传递的数据结构（一个 TypedDict）。
2. `triage_router` 节点：读取 `email_input`，用 `llm_router` 判断这封邮件该 ignore / notify / respond，
   然后通过返回 `Command(goto=..., update=...)` 来**动态决定下一步去哪个节点**（这就是 LangGraph 里的
   "条件路由 / conditional routing"，比起画死的边更灵活）。
3. `response_agent` 节点：即上面构建的 ReAct agent，只有分类结果是 "respond" 时才会被路由过去执行。

In [ ]:
# add_messages 是 LangGraph 提供的 reducer：当多个更新同时写入 "messages" 字段时，
# 它会做"追加"而不是"覆盖"（这是维护多轮对话历史的标准写法）。
from langgraph.graph import add_messages

# TODO: 请在此处补全代码
# 定义 State(TypedDict)，需要两个字段：
#   - email_input: dict            原始待处理邮件
#   - messages: Annotated[list, add_messages]   对话历史，用 add_messages 做"追加"而非"覆盖"
class State(TypedDict):
    pass

In [ ]:
# StateGraph/START/END：构建图的核心 API
# Command：节点函数除了返回"状态更新"，还可以返回 Command 来指定"下一步去哪个节点"（动态路由）
from langgraph.graph import StateGraph,START,END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image,display

In [ ]:
# triage_router：图的入口节点。
# 返回类型标注 Command[Literal["response_agent","__end__"]] 表示：这个节点自己决定
# 图接下来走到 "response_agent" 还是直接结束（"__end__"），而不是靠预先画好的固定边。
#
# TODO: 请在此处补全代码
# 实现 triage_router(state) -> Command[Literal["response_agent","__end__"]]：
#   1. 从 state['email_input'] 里取出 author/to/subject/email_thread
#   2. 用 triage_system_prompt / triage_user_prompt 拼出 system_prompt / user_prompt
#   3. 调用 llm_router.invoke([...]) 得到分类结果 result
#   4. 根据 result.classification 分三种情况：
#        "respond" -> goto="response_agent"，update 里塞一条 user 消息
#                     （内容类似 f"Respond to the email {state['email_input']}"）
#        "ignore"  -> goto=END，update=None
#        "notify"  -> goto=END，update=None（现实中这里通常会做提醒通知）
#        其他值    -> raise ValueError
#   5. 别忘了最后 return Command(goto=goto, update=update)
#      （提示：漏掉这一步 return，图会在这个节点后静默结束，不会报错但结果不对）
def triage_router(state:State)->Command[
    Literal["response_agent","__end__"]
]:
    pass

## Put it all together

In [ ]:
# 组装整张图：
# 1. add_node(triage_router) 不传字符串名字时，LangGraph 会用函数名 "triage_router" 作为节点名
# 2. add_node("response_agent", agent) 把上面构建的 ReAct agent 注册为节点
# 3. 只需要一条固定边 START -> triage_router；triage_router -> response_agent / END
#    是靠它返回的 Command 动态决定的，不需要再手动 add_edge
#
# TODO: 请在此处补全代码
# 用 StateGraph(State) 创建图，依次:
#   add_node(triage_router)
#   add_node("response_agent", agent)
#   add_edge(START, "triage_router")
#   最后 compile() 得到可执行的 email_agent
email_agent = None

In [ ]:
# 可视化整张图结构（Mermaid 图）
# 注意：draw_mermaid_png() 默认会请求 mermaid.ink 的在线服务来渲染图片，
# 需要真实的互联网访问；如果环境没有联网，这一步会因为网络请求失败而报错，
# 与 API key 无关，属于预期的环境限制，不是代码逻辑问题。
display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# 测试用例 1：一封营销垃圾邮件，预期被分类为 "ignore"，图应直接结束
email_input = {
    "author": "Marketing Team <marketing@amazingdeals.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "🔥 EXCLUSIVE OFFER: Limited Time Discount on Developer Tools! 🔥",
    "email_thread": """Dear Valued Developer,

Don't miss out on this INCREDIBLE opportunity!

🚀 For a LIMITED TIME ONLY, get 80% OFF on our Premium Developer Suite!

✨ FEATURES:
- Revolutionary AI-powered code completion
- Cloud-based development environment
- 24/7 customer support
- And much more!

💰 Regular Price: $999/month
🎉 YOUR SPECIAL PRICE: Just $199/month!

🕒 Hurry! This offer expires in:
24 HOURS ONLY!

Click here to claim your discount: https://amazingdeals.com/special-offer

Best regards,
Marketing Team
---
To unsubscribe, click here
""",
}

In [ ]:
# 跑一次完整的图：会先进 triage_router 节点（真实网络请求，假 key 场景下预期报错）
response = email_agent.invoke({"email_input": email_input})

In [ ]:
# 测试用例 2：一封需要回复的邮件，预期分类为 "respond"，会路由到 response_agent 继续处理
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [ ]:
# 修复 triage_router 缺失的 return 之后，这里应该能观察到：分类为 respond -> 路由到
# response_agent -> response_agent 调用工具/生成回复 -> messages 里出现多轮记录
response = email_agent.invoke({"email_input": email_input})

In [ ]:
# 逐条打印完整对话历史（触发 triage 的 user 消息 + agent 的推理/工具调用/最终回复）
for m in response["messages"]:
    m.pretty_print()